# Station Stacking v17 - KDAL

Experimental notebook for `KDAL`.

V17 is the importance-pruned experiment: only the KATL v16 fused features with `max_importance_mae_f >= 0.015`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TIMING_MODE = "same_day_11am_live_safe"
TARGET_SOURCE = "iem_hourly"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
EXPORT_MODEL_WEIGHTS = True
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v17"
V15_OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v15"
V16_OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v16"
MODEL_VERSION = "station_high_regressor_v17_importance_015_stack"

PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from scripts.run_station_stacking_v17 import write_reference_comparisons
from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V17_ADDITIONAL_FEATURE_COLUMNS,
    V17_DROPPED_FEATURE_COLUMNS,
    V17_IMPORTANCE_015_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V17 Contract

`feature_version="v17_importance_015"` uses only the 17-feature allowlist from the KATL v16 fused importance summary. The single v13 weather interaction is kept only when train-year coverage passes.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

{
    "folds": fold_spec,
    "v17_feature_count": len(V17_IMPORTANCE_015_FEATURE_COLUMNS),
    "v17_features": V17_IMPORTANCE_015_FEATURE_COLUMNS,
    "v17_coverage_gated_features": V17_ADDITIONAL_FEATURE_COLUMNS,
    "v17_dropped_features": sorted(V17_DROPPED_FEATURE_COLUMNS),
}


{'folds':                      fold  train_start_year  train_end_year  validation_year
 0  fold_2021_2023_to_2024              2021            2023             2024
 1  fold_2021_2024_to_2025              2021            2024             2025,
 'v17_feature_count': 17,
 'v17_features': ['nbm_high_minus_observed_high_temp_f',
  'v8_provider_median_remaining_from_high_so_far_f',
  'observed_high_so_far_change_since_9am_f',
  'nbm_high_minus_observed_temp_f',
  'v3_remaining_warmup_per_spread_f',
  'v8_provider_max_remaining_from_high_so_far_f',
  'v8_provider_mean_remaining_vs_month_normal_f',
  'hrrr_high_minus_observed_high_temp_f',
  'observed_high_temp_minus_temp_at_as_of_f',
  'gfs_high_minus_observed_high_temp_f',
  'observed_cloud_cover_at_as_of',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_humidity_remaining_warmup_interaction',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'gfs_high_minus_observed_temp_f',
  'hrrr_rolling_bias_30d_f',
  'observed_temp_change_l

## Data Availability


In [4]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1992,2021-01-01,2026-06-21
7,KDAL,hrrr,1998,2021-01-01,2026-06-21
8,KDAL,nbm,1997,2021-01-01,2026-06-21


## Run Importance-Pruned Model


In [5]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v17_importance_015",
    target_mode="remaining_warmup",
    target_source=TARGET_SOURCE,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=OUTPUT_DIR / "importance_015",
    climatology_normals_path=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9" / "station_rolling_10y_daily_high_normals.csv",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/KDAL_optuna.sqlite3')

In [6]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:2711: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2710: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2711: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,730,1.509339,2.107499
1,validation_2024_2025,lightgbm,730,1.518386,2.113201
2,validation_2024_2025,catboost,730,1.561859,2.136072
3,validation_2024_2025,provider_mean,730,3.178604,3.977779
4,validation_2024_2025,provider_median,730,2.827166,3.671639
5,validation_2024_2025,nbm_raw,730,2.619924,3.489639
6,validation_2024_2025,hrrr_raw,730,5.584943,6.432032
7,validation_2024_2025,gfs_raw,730,2.717947,3.702726
8,test_2026,xgboost,172,1.602308,2.038753
9,test_2026,lightgbm,172,1.626183,2.043363


In [7]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v17",
    )
    display((exported_weights.bundle_path, exported_weights.manifest_path))

write_reference_comparisons(OUTPUT_DIR, V15_OUTPUT_DIR, V16_OUTPUT_DIR, STATION_ID)


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/model_weights/KDAL_station_high_regressor_v17_importance_015_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/model_weights/KDAL_station_high_regressor_v17_importance_015_stack.json'))

Wrote KDAL v17 reference comparison: D:\dev\weather-research\data\calibration\station_stacking_v17\KDAL_v17_importance_015_vs_references_common_date_comparison.csv


## Reference Comparison


In [8]:
comparison_path = OUTPUT_DIR / f"{STATION_ID}_v17_importance_015_vs_references_common_date_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
comparison.sort_values(["method", "delta_mae_f", "reference"]) if not comparison.empty else comparison


EmptyDataError: No columns to parse from file

## Selected Feature Audit


In [9]:
expected = set(V17_IMPORTANCE_015_FEATURE_COLUMNS)
selected = set(result.feature_columns["feature"].astype(str))
{
    "selected_feature_count": len(selected),
    "expected_feature_count": len(expected),
    "selected_features": sorted(selected),
    "missing_expected": sorted(expected - selected),
    "unexpected_selected": sorted(selected - expected),
    "coverage_gated_selected": sorted(selected & set(V17_ADDITIONAL_FEATURE_COLUMNS)),
}


{'selected_feature_count': 17,
 'expected_feature_count': 17,
 'selected_features': ['gfs_high_minus_observed_high_temp_f',
  'gfs_high_minus_observed_temp_f',
  'hrrr_high_minus_observed_high_temp_f',
  'hrrr_rolling_bias_30d_f',
  'nbm_high_minus_observed_high_temp_f',
  'nbm_high_minus_observed_temp_f',
  'observed_cloud_cover_at_as_of',
  'observed_high_so_far_change_since_9am_f',
  'observed_high_temp_minus_temp_at_as_of_f',
  'observed_temp_change_last_3h_f',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'v3_humidity_remaining_warmup_interaction',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_remaining_warmup_per_spread_f',
  'v8_provider_max_remaining_from_high_so_far_f',
  'v8_provider_mean_remaining_vs_month_normal_f',
  'v8_provider_median_remaining_from_high_so_far_f'],
 'missing_expected': [],
 'unexpected_selected': [],
 'coverage_gated_selected': ['v13_forecast_temp_bias_remaining_warmup_interaction']}

## Accidental Weather Sprawl Check


In [10]:
raw_weather_tokens = (
    "cloud",
    "ceiling",
    "dewpoint",
    "forecast_temp_at_as_of",
    "humidity",
    "precip",
    "pressure",
    "shortwave",
    "visibility",
    "wind_",
)
selected_features = result.feature_columns["feature"].astype(str)
accidental_weather_selected = pd.DataFrame(
    [
        {"feature": feature}
        for feature in selected_features
        if (
            feature.startswith(("gfs_", "hrrr_", "nbm_", "v13_", "v8_"))
            and any(token in feature for token in raw_weather_tokens)
            and feature not in set(V17_IMPORTANCE_015_FEATURE_COLUMNS)
        )
    ]
)

accidental_weather_selected


""


## Feature Importance


In [11]:
importance_path = config.resolved_output_dir() / f"{STATION_ID}_year_split_feature_importance.csv"
importance = pd.read_csv(importance_path) if importance_path.exists() else pd.DataFrame()
importance.sort_values(["method", "importance_mae_f"], ascending=[True, False]) if not importance.empty else importance


KeyError: 'importance_mae_f'

## Rounded Within 1F Accuracy


In [12]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)
predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
8,oof_2026,xgboost,172,99,57.558140
7,oof_2026,ridge_stack,172,98,56.976744
0,oof_2026,catboost,172,97,56.395349
3,oof_2026,lightgbm,172,92,53.488372
4,oof_2026,nbm_raw,172,76,44.186047
6,oof_2026,provider_median,172,54,31.395349
1,oof_2026,gfs_raw,172,53,30.813953
5,oof_2026,provider_mean,172,43,25.000000
2,oof_2026,hrrr_raw,172,15,8.720930
12,validation_2024_2025,lightgbm,730,454,62.191781


## Bracket Metrics


In [13]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,172,1.602308,2.038753,40.697674
1,lightgbm,172,1.626183,2.043363,37.790698
2,catboost,172,1.659799,2.114411,36.627907
3,ridge_stack,172,1.577292,2.007893,38.372093
4,provider_mean,172,3.194053,3.954456,19.186047
5,provider_median,172,2.892289,3.729196,19.767442
6,nbm_raw,172,2.353419,3.203275,30.813953
7,hrrr_raw,172,5.595144,6.373455,5.232558
8,gfs_raw,172,2.914622,3.791390,22.674419
